# Series Data Collection 

## Objective

The objective of this notebook is to collect information on series adaptations from Amazon Prime Video and Netflix and enrich the dataset with IMDb metadata for further analysis.

## Data Sources

* Wikipedia pages containing lists of Netflix/Amazon original titles.
* OMDb API for IMDb metadata such as ratings, vote counts, genres, release years, and content ratings.

## Methodology

1. Titles were collected through web scraping from Wikipedia.
2. IMDb metadata was retrieved using the OMDb API.
3. Each title was manually verified to determine whether it was based on a book.
4. Only confirmed book adaptations were retained for further analysis.


# Amazon Series Data Collection

## Web Scraping Wikipedia Data

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
url = "https://en.wikipedia.org/wiki/List_of_Amazon_Prime_Video_original_programming"

headers = {
    "User-Agent": "Mozilla/5.0"
}

html = requests.get(url, headers=headers).text

In [3]:
from io import StringIO

tables = pd.read_html(StringIO(html))

print(len(tables))

87


In [4]:
for i, table in enumerate(tables):
    print(f"\nTABLE {i}")
    print(table.head())


TABLE 0
                                       Title         Genre           Premiere  \
0                                    Reacher   Crime drama   February 4, 2022   
1                          The Terminal List  Action drama       July 1, 2022   
2  The Lord of the Rings: The Rings of Power       Fantasy  September 1, 2022   
3                           The Devil's Hour      Thriller   October 28, 2022   
4                                    Citadel     Spy drama     April 28, 2023   

                  Seasons                                             Status  
0  3 seasons, 24 episodes                      Renewed for seasons 4–5[2][3]  
1    1 season, 8 episodes  Season 2 due to premiere on October 21, 2026[4...  
2  2 seasons, 16 episodes  Season 3 due to premiere on November 11, 2026[...  
3  2 seasons, 11 episodes                                         Renewed[8]  
4  2 seasons, 13 episodes                                            Pending  

TABLE 1
              Title  

In [5]:
dfs = []

for table in tables:
    if "Title" in table.columns:
        dfs.append(table)

amazon_df = pd.concat(dfs, ignore_index=True)

amazon_df.drop_duplicates(subset=["Title"], inplace=True)

print(amazon_df.head())
print(len(amazon_df))

                                       Title         Genre           Premiere  \
0                                    Reacher   Crime drama   February 4, 2022   
1                          The Terminal List  Action drama       July 1, 2022   
2  The Lord of the Rings: The Rings of Power       Fantasy  September 1, 2022   
3                           The Devil's Hour      Thriller   October 28, 2022   
4                                    Citadel     Spy drama     April 28, 2023   

                  Seasons                                             Status  \
0  3 seasons, 24 episodes                      Renewed for seasons 4–5[2][3]   
1    1 season, 8 episodes  Season 2 due to premiere on October 21, 2026[4...   
2  2 seasons, 16 episodes  Season 3 due to premiere on November 11, 2026[...   
3  2 seasons, 11 episodes                                         Renewed[8]   
4  2 seasons, 13 episodes                                            Pending   

  Language Subject Partner Prev.

In [6]:
amazon_df.info()

<class 'pandas.DataFrame'>
Index: 264 entries, 0 to 268
Data columns (total 11 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   Title                         264 non-null    str   
 1   Genre                         245 non-null    str   
 2   Premiere                      207 non-null    object
 3   Seasons                       207 non-null    str   
 4   Status                        141 non-null    str   
 5   Language                      83 non-null     str   
 6   Subject                       19 non-null     str   
 7   Partner                       4 non-null      str   
 8   Prev. network(s)              4 non-null      str   
 9   Prime Video exclusive region  16 non-null     str   
 10  Partner/Country               3 non-null      str   
dtypes: object(1), str(10)
memory usage: 24.8+ KB


In [7]:
amazon_df = amazon_df[["Title", "Premiere"]]

In [8]:
amazon_df.info()

<class 'pandas.DataFrame'>
Index: 264 entries, 0 to 268
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Title     264 non-null    str   
 1   Premiere  207 non-null    object
dtypes: object(1), str(1)
memory usage: 6.2+ KB


In [9]:
print(amazon_df["Title"].duplicated().sum())

0


In [10]:
amazon_df.sample(n=10, random_state=42)

,Title,Premiere
71,"Mentiras, The Series","June 13, 2025"
199,Glamsquad[133][134],TBA
85,Clarkson's Farm,"June 11, 2021"
252,She-Ra[176],NaN
113,LOL: In Real Life!,"November 21, 2025"
123,Dish It Out,"September 5, 2025"
117,American Gladiators,"April 17, 2026"
183,Lore Olympus[122],TBA
9,Ballard,"July 9, 2025"
256,Taylor Lautner: Werewolf Hunter[180],NaN


In [11]:
amazon_df['Title'] = amazon_df['Title'].str.replace(r'\[\d+\]', '', regex=True).str.strip()
amazon_df['Premiere'] = amazon_df['Premiere'].str.replace(r'\[\d+\]', '', regex=True).str.strip()

In [12]:
amazon_df["Title"] = (amazon_df["Title"].str.replace(r"\[[a-z]\]", "", regex=True).str.strip())

In [13]:
amazon_df[amazon_df["Title"].str.contains(r"\[", regex=True, na=False)]

,Title,Premiere


In [14]:
amazon_df.sample(n=10, random_state=42)

,Title,Premiere
71,"Mentiras, The Series","June 13, 2025"
199,Glamsquad,TBA
85,Clarkson's Farm,"June 11, 2021"
252,She-Ra,NaN
113,LOL: In Real Life!,"November 21, 2025"
123,Dish It Out,"September 5, 2025"
117,American Gladiators,"April 17, 2026"
183,Lore Olympus,TBA
9,Ballard,"July 9, 2025"
256,Taylor Lautner: Werewolf Hunter,NaN


In [15]:
amazon_df['Premiere'] = pd.to_datetime(amazon_df['Premiere'],errors='coerce')

In [16]:
cutoff_date = pd.Timestamp('2026-06-18')

In [17]:
premiered_df = amazon_df[(amazon_df['Premiere'].notna()) & (amazon_df['Premiere'] < cutoff_date)].copy()

premiered_df.to_csv('../data/series/raw/amazon_titles.csv', index=False)

In [18]:
upcoming_tba_df = amazon_df[(amazon_df['Premiere'].isna()) | (amazon_df['Premiere'] >= cutoff_date)].copy()

upcoming_tba_df.to_csv('../data/series/raw/upcoming_amazon.csv', index=False)

In [19]:
print(f"Premiered titles: {len(premiered_df)}")
print(f"Upcoming/TBA titles: {len(upcoming_tba_df)}")

Premiered titles: 128
Upcoming/TBA titles: 136


## Retrieving IMDb Metadata

In [20]:
import requests

API_KEY = "adebd275"

title = "Reacher"

url = f"https://www.omdbapi.com/?t={title}&apikey={API_KEY}"

response = requests.get(url)

data = response.json()

print(data)

{'Title': 'Reacher', 'Year': '2022–', 'Rated': 'TV-MA', 'Released': '04 Feb 2022', 'Runtime': '1 min', 'Genre': 'Action, Crime, Drama', 'Director': 'N/A', 'Writer': 'Nick Santora', 'Actors': 'Alan Ritchson, Maria Sten, Malcolm Goodwin', 'Plot': 'Itinerant former military policeman Jack Reacher solves crimes and metes out his own brand of street justice. Based on the novels by Lee Child.', 'Language': 'English', 'Country': 'United States', 'Awards': '3 wins & 13 nominations total', 'Poster': 'https://m.media-amazon.com/images/M/MV5BMzdjYWZlMDQtYzdhNi00NmRlLTg2NzUtMTI3MWFhZDliNjBiXkEyXkFqcGc@._V1_QL75_UX380_CR0,57,380,562_.jpg', 'Ratings': [{'Source': 'Internet Movie Database', 'Value': '8.0/10'}], 'Metascore': 'N/A', 'imdbRating': '8.0', 'imdbVotes': '291,841', 'imdbID': 'tt9288030', 'Type': 'series', 'totalSeasons': '4', 'Response': 'True'}


In [21]:
def get_omdb_data(title):

    try:
        url = f"https://www.omdbapi.com/?t={title}&apikey={API_KEY}"

        data = requests.get(url).json()

        return pd.Series({
            "Rated": data.get("Rated"),
            "imdb_rating": data.get("imdbRating"),
            "imdb_votes": data.get("imdbVotes"),
            "imdb_genre": data.get("Genre"),
            "imdb_year": data.get("Year"),
            "imdb_type": data.get("Type")
        })

    except:
        return pd.Series()

In [22]:
amazon_df = pd.read_csv('../data/series/raw/amazon_titles.csv')

In [23]:
omdb_data = amazon_df["Title"].apply(get_omdb_data)

amazon_df = pd.concat(
    [amazon_df, omdb_data],
    axis=1
)

In [24]:
amazon_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Title        128 non-null    str  
 1   Premiere     128 non-null    str  
 2   Rated        109 non-null    str  
 3   imdb_rating  109 non-null    str  
 4   imdb_votes   109 non-null    str  
 5   imdb_genre   109 non-null    str  
 6   imdb_year    109 non-null    str  
 7   imdb_type    109 non-null    str  
dtypes: str(8)
memory usage: 8.1 KB


In [25]:
amazon_df.head()

,Title,Premiere,Rated,imdb_rating,imdb_votes,imdb_genre,imdb_year,imdb_type
0,Reacher,2022-02-04,TV-MA,8.0,"291,841","Action, Crime, Drama",2022–,series
1,The Terminal List,2022-07-01,TV-MA,7.9,"127,044","Action, Drama, Thriller",2022–,series
2,The Lord of the Rings: The Rings of Power,2022-09-01,TV-14,6.9,"436,113","Action, Adventure, Drama",2022–,series
3,The Devil's Hour,2022-10-28,TV-MA,7.6,"32,008","Drama, Mystery, Thriller",2022–,series
4,Citadel,2023-04-28,TV-14,6.2,"37,316","Action, Drama, Thriller",2023–,series


In [26]:
amazon_df['imdb_votes'] = (amazon_df['imdb_votes'].astype(str).str.replace(',', '', regex=False))

In [27]:
amazon_df['imdb_rating'] = pd.to_numeric(amazon_df['imdb_rating'], errors='coerce')
amazon_df['imdb_votes'] = pd.to_numeric(amazon_df['imdb_votes'], errors='coerce')

In [28]:
amazon_df['imdb_year'] = (amazon_df['imdb_year'].astype(str).str.extract(r'(\d{4})')[0]).astype('Int64')

amazon_df['imdb_year'] = pd.to_numeric(amazon_df['imdb_year'], errors='coerce')

In [29]:
amazon_df['imdb_rating'].isna().sum()

np.int64(48)

In [30]:
amazon_df = amazon_df.dropna(subset=['imdb_rating'])

In [31]:
amazon_df.head()

,Title,Premiere,Rated,imdb_rating,imdb_votes,imdb_genre,imdb_year,imdb_type
0,Reacher,2022-02-04,TV-MA,8.0,291841.0,"Action, Crime, Drama",2022,series
1,The Terminal List,2022-07-01,TV-MA,7.9,127044.0,"Action, Drama, Thriller",2022,series
2,The Lord of the Rings: The Rings of Power,2022-09-01,TV-14,6.9,436113.0,"Action, Adventure, Drama",2022,series
3,The Devil's Hour,2022-10-28,TV-MA,7.6,32008.0,"Drama, Mystery, Thriller",2022,series
4,Citadel,2023-04-28,TV-14,6.2,37316.0,"Action, Drama, Thriller",2023,series


In [32]:
amazon_df.info()

<class 'pandas.DataFrame'>
Index: 80 entries, 0 to 125
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Title        80 non-null     str    
 1   Premiere     80 non-null     str    
 2   Rated        80 non-null     str    
 3   imdb_rating  80 non-null     float64
 4   imdb_votes   80 non-null     float64
 5   imdb_genre   80 non-null     str    
 6   imdb_year    80 non-null     Int64  
 7   imdb_type    80 non-null     str    
dtypes: Int64(1), float64(2), str(5)
memory usage: 5.7 KB


In [33]:
amazon_df.shape

(80, 8)

In [34]:
amazon_df.to_csv(
    "../data/series/processed/amazon_imdb.csv",
    index=False
)

## Adaptation Verification and Saving the Dataset

In [35]:
import pandas as pd

df = pd.read_csv("../data/series/processed/amazon_imdb_adaptations.csv")

df.head()

,Title,Premiere,Rated,imdb_rating,imdb_votes,imdb_genre,imdb_year,imdb_type,is_book_adaptation,book_title,book_author
0,Reacher,04/02/2022,TV-MA,8.0,291841.0,"Action, Crime, Drama",2022,series,y,Jack Reacher series,Lee Child
1,The Terminal List,01/07/2022,TV-MA,7.9,127044.0,"Action, Drama, Thriller",2022,series,y,The Terminal List,Jack Carr
2,The Lord of the Rings: The Rings of Power,01/09/2022,TV-14,6.9,436113.0,"Action, Adventure, Drama",2022,series,y,The Lord of the Rings and related Middle-earth...,J. R. R. Tolkien
3,The Devil's Hour,28/10/2022,TV-MA,7.6,32008.0,"Drama, Mystery, Thriller",2022,series,n,NaN,NaN
4,Citadel,28/04/2023,TV-14,6.2,37316.0,"Action, Drama, Thriller",2023,series,n,NaN,NaN


In [36]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               80 non-null     str    
 1   Premiere            80 non-null     str    
 2   Rated               48 non-null     str    
 3   imdb_rating         80 non-null     float64
 4   imdb_votes          80 non-null     float64
 5   imdb_genre          79 non-null     str    
 6   imdb_year           80 non-null     int64  
 7   imdb_type           80 non-null     str    
 8   is_book_adaptation  80 non-null     str    
 9   book_title          20 non-null     str    
 10  book_author         20 non-null     str    
dtypes: float64(2), int64(1), str(8)
memory usage: 7.0 KB


In [37]:
df.columns

Index(['Title', 'Premiere', 'Rated', 'imdb_rating', 'imdb_votes', 'imdb_genre',
       'imdb_year', 'imdb_type', 'is_book_adaptation', 'book_title',
       'book_author'],
      dtype='str')

In [38]:
df["is_book_adaptation"].value_counts()

is_book_adaptation
n    60
y    20
Name: count, dtype: int64

In [39]:
adaptations_df = df[df["is_book_adaptation"] == "y"].copy()

In [40]:
adaptations_df.shape

(20, 11)

In [41]:
adaptations_df[["book_title", "book_author"]].isnull().sum()

book_title     0
book_author    0
dtype: int64

In [42]:
adaptations_df["book_title"] = adaptations_df["book_title"].str.strip().str.lower()
adaptations_df["book_author"] = adaptations_df["book_author"].str.strip().str.lower()

In [43]:
adaptations_df[["book_title", "book_author"]].head(10)

,book_title,book_author
0,jack reacher series,lee child
1,the terminal list,jack carr
2,the lord of the rings and related middle-earth...,j. r. r. tolkien
6,alex cross series,james patterson
8,we were liars,e. lockhart
9,renée ballard novels,michael connelly
10,the terminal list universe,jack carr
11,malice,keigo higashino
13,56 days,catherine ryan howard
14,young sherlock holmes series,andrew lane


In [44]:
import re

def clean_title(text):
    text = text.lower()
    text = re.sub(r"\(.*?\)", "", text)   # remove brackets content
    text = text.replace(",", " ")
    text = re.sub(r"\s+", " ", text)     # remove extra spaces
    return text.strip()

adaptations_df["book_title"] = adaptations_df["book_title"].apply(clean_title)

In [45]:
def to_search_key(text):
    text = text.lower()
    text = re.sub(r"(series|universe|novels|books)", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

adaptations_df["search_key"] = adaptations_df["book_title"].apply(to_search_key)

In [46]:
adaptations_df.head(10)

,Title,Premiere,Rated,imdb_rating,imdb_votes,imdb_genre,imdb_year,imdb_type,is_book_adaptation,book_title,book_author,search_key
0,Reacher,04/02/2022,TV-MA,8.0,291841.0,"Action, Crime, Drama",2022,series,y,jack reacher series,lee child,jack reacher
1,The Terminal List,01/07/2022,TV-MA,7.9,127044.0,"Action, Drama, Thriller",2022,series,y,the terminal list,jack carr,the terminal list
2,The Lord of the Rings: The Rings of Power,01/09/2022,TV-14,6.9,436113.0,"Action, Adventure, Drama",2022,series,y,the lord of the rings and related middle-earth...,j. r. r. tolkien,the lord of the rings and related middle-earth...
6,Cross,14/11/2024,TV-MA,7.2,34967.0,"Action, Crime, Drama",2024,series,y,alex cross series,james patterson,alex cross
8,We Were Liars,18/06/2025,TV-14,6.8,15550.0,"Drama, Mystery, Romance",2025,series,y,we were liars,e. lockhart,we were liars
9,Ballard,09/07/2025,TV-14,7.5,22736.0,"Crime, Drama",2025,series,y,renée ballard novels,michael connelly,renée ballard
10,The Terminal List: Dark Wolf,27/08/2025,TV-MA,7.6,26656.0,"Action, Drama, Thriller",2025,series,y,the terminal list universe,jack carr,the terminal list
11,Malice,14/11/2025,R,6.5,30561.0,"Crime, Mystery, Thriller",1993,movie,y,malice,keigo higashino,malice
13,56 Days,18/02/2026,NaN,6.5,1393.0,"Crime, Drama, Mystery",2026,series,y,56 days,catherine ryan howard,56 days
14,Young Sherlock,04/03/2026,TV-14,7.5,24793.0,"Action, Adventure, Drama",2026,series,y,young sherlock holmes series,andrew lane,young sherlock holmes


In [47]:
adaptations_df.duplicated().sum()

np.int64(0)

In [48]:
adaptations_df.info()

<class 'pandas.DataFrame'>
Index: 20 entries, 0 to 60
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               20 non-null     str    
 1   Premiere            20 non-null     str    
 2   Rated               17 non-null     str    
 3   imdb_rating         20 non-null     float64
 4   imdb_votes          20 non-null     float64
 5   imdb_genre          20 non-null     str    
 6   imdb_year           20 non-null     int64  
 7   imdb_type           20 non-null     str    
 8   is_book_adaptation  20 non-null     str    
 9   book_title          20 non-null     str    
 10  book_author         20 non-null     str    
 11  search_key          20 non-null     str    
dtypes: float64(2), int64(1), str(9)
memory usage: 2.0 KB


In [49]:
mismatch_df = adaptations_df[pd.to_datetime(adaptations_df['Premiere'], dayfirst=True).dt.year != adaptations_df['imdb_year']]

print(len(mismatch_df))

3


In [50]:
mismatch_df['year_diff'] = (pd.to_datetime(mismatch_df['Premiere'], dayfirst=True).dt.year - mismatch_df['imdb_year']).abs()

print(mismatch_df['year_diff'].value_counts().sort_index())

year_diff
1     1
32    1
33    1
Name: count, dtype: int64


In [51]:
mismatch_df = mismatch_df.sort_values('year_diff', ascending=False)

mismatch_df.to_csv('../data/series/processed/amazon_year_mismatches.csv', index=False)

print(f"Saved {len(mismatch_df)} mismatches")

Saved 3 mismatches


In [52]:
mismatch_df = pd.read_csv('../data/series/processed/corrected_amazon_mismatches.csv')

print("Remove:", (mismatch_df['keep'].str.lower() == 'n').sum())
print("Update:", (mismatch_df['keep'].str.lower() == 'y').sum())

Remove: 1
Update: 2


In [53]:
titles_to_remove = mismatch_df.loc[mismatch_df['keep'].str.lower() == 'n','Title']

adaptations_df = adaptations_df[~adaptations_df['Title'].isin(titles_to_remove)].copy()

In [54]:
updates = mismatch_df[mismatch_df['keep'].str.lower() == 'y'].set_index('Title')

adaptations_df = adaptations_df.set_index('Title')

cols_to_update = [
    'Rated',
    'imdb_rating',
    'imdb_votes',
    'imdb_genre',
    'imdb_year',
    'imdb_type'
]

for col in cols_to_update:
    adaptations_df.loc[updates.index, col] = updates[col]

adaptations_df = adaptations_df.reset_index()

In [55]:
print("Final rows:", len(adaptations_df))

Final rows: 19


In [56]:
adaptations_df.to_csv("../data/series/final/amazon_series.csv", index=False)

# Netflix Series Data Collection

## Web Scraping Wikipedia Data

In [57]:
import pandas as pd
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/List_of_Netflix_original_programming"

headers = {
    "User-Agent": "Mozilla/5.0"
}

html = requests.get(url, headers=headers).text

tables = pd.read_html(StringIO(html))

print(len(tables))

112


In [58]:
for i, table in enumerate(tables):
    print(f"\nTABLE {i}")
    print(table.head())


TABLE 0
             Title                         Genre           Premiere  \
0     Virgin River                Romantic drama   December 6, 2019   
1      The Witcher                 Fantasy drama  December 20, 2019   
2      Outer Banks           Coming-of-age drama     April 15, 2020   
3  Sweet Magnolias                Romantic drama       May 19, 2020   
4       Bridgerton  Alternate historical romance  December 25, 2020   

                  Seasons    Runtime  \
0  7 seasons, 74 episodes  39–54 min   
1  4 seasons, 32 episodes  47–67 min   
2  4 seasons, 40 episodes  40–85 min   
3  5 seasons, 50 episodes  43–54 min   
4  4 seasons, 32 episodes  52–73 min   

                                              Status  
0                                         Renewed[1]  
1         Final season due to premiere in 2026[2][3]  
2  Final season due to premiere on August 20, 202...  
3                                            Pending  
4                         Renewed for seasons 5–

In [59]:
for i, table in enumerate(tables):
    if "Title" in table.columns:
        print(f"Table {i}")
        print(table.columns.tolist())
        print()

Table 0
['Title', 'Genre', 'Premiere', 'Seasons', 'Runtime', 'Status']

Table 1
['Title', 'Genre', 'Premiere', 'Seasons', 'Runtime', 'Status']

Table 2
['Title', 'Genre', 'Premiere', 'Seasons', 'Runtime', 'Status']

Table 3
['Title', 'Genre', 'Premiere', 'Seasons', 'Runtime', 'Language', 'Status']

Table 4
['Title', 'Genre', 'Premiere', 'Seasons', 'Runtime', 'Language', 'Status']

Table 5
['Title', 'Genre', 'Premiere', 'Seasons', 'Runtime', 'Language', 'Status']

Table 6
['Title', 'Genre', 'Premiere', 'Seasons', 'Runtime', 'Status']

Table 7
['Title', 'Genre', 'Premiere', 'Seasons', 'Runtime', 'Status']

Table 8
['Title', 'Genre', 'Premiere', 'Seasons', 'Runtime', 'Status']

Table 9
['Title', 'Genre', 'Premiere', 'Seasons', 'Runtime', 'Status']

Table 10
['Title', 'Genre', 'Premiere', 'Seasons', 'Runtime', 'Status']

Table 11
['Title', 'Genre', 'Premiere', 'Seasons', 'Runtime', 'Status']

Table 12
['Title', 'Genre', 'Premiere', 'Seasons', 'Runtime', 'Status']

Table 13
['Title', 'Genre

In [60]:
df = []

needed = ["Title", "Premiere"]

for table in tables:
    table.columns = [
        col[0] if isinstance(col, tuple) else col
        for col in table.columns
    ]

    if all(col in table.columns for col in needed):
        df.append(table[needed])

netflix_df = pd.concat(df, ignore_index=True)

netflix_df.drop_duplicates(subset=["Title"], inplace=True)

print(netflix_df.head())
print(netflix_df.shape)

             Title           Premiere
0     Virgin River   December 6, 2019
1      The Witcher  December 20, 2019
2      Outer Banks     April 15, 2020
3  Sweet Magnolias       May 19, 2020
4       Bridgerton  December 25, 2020
(532, 2)


In [61]:
netflix_df.info()

<class 'pandas.DataFrame'>
Index: 532 entries, 0 to 542
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Title     532 non-null    str   
 1   Premiere  532 non-null    object
dtypes: object(1), str(1)
memory usage: 12.5+ KB


In [62]:
netflix_df.isna().sum()

Title       0
Premiere    0
dtype: int64

In [63]:
netflix_df = netflix_df.drop_duplicates()

In [64]:
netflix_df.shape

(532, 2)

In [65]:
netflix_df.sample(n=10, random_state=42)

,Title,Premiere
6,The Lincoln Lawyer,"May 13, 2022"
499,#Love[317],2026[77]
108,Delhi Crime,"March 22, 2019"
497,Empress of Flames[316],2026
459,Anaesthesia[293],TBA
231,The Echoes of Survivors: Inside Korea's Tragedies,"August 15, 2025"
246,The Dinosaurs,"March 6, 2026"
444,Beauty in the Beast[284],TBA
358,The God of the Woods[197],TBA
74,Dandelion,"April 16, 2026"


In [66]:
netflix_df['Title'] = netflix_df['Title'].str.replace(r'\[\d+\]', '', regex=True).str.strip()
netflix_df['Premiere'] = netflix_df['Premiere'].str.replace(r'\[\d+\]', '', regex=True).str.strip()

In [67]:
netflix_df["Title"] = (netflix_df["Title"].str.replace(r"\[[a-z]\]", "", regex=True).str.strip())

In [68]:
netflix_df[netflix_df["Title"].str.contains(r"\[", regex=True, na=False)]

,Title,Premiere


In [69]:
netflix_df.sample(n=10, random_state=42)

,Title,Premiere
6,The Lincoln Lawyer,"May 13, 2022"
499,#Love,2026
108,Delhi Crime,"March 22, 2019"
497,Empress of Flames,2026
459,Anaesthesia,TBA
231,The Echoes of Survivors: Inside Korea's Tragedies,"August 15, 2025"
246,The Dinosaurs,"March 6, 2026"
444,Beauty in the Beast,TBA
358,The God of the Woods,TBA
74,Dandelion,"April 16, 2026"


In [70]:
netflix_df['Premiere'] = pd.to_datetime(netflix_df['Premiere'],errors='coerce')

In [71]:
cutoff_date = pd.Timestamp('2026-06-18')

In [72]:
premiered_df = netflix_df[(netflix_df['Premiere'] .notna()) & (netflix_df['Premiere'] < cutoff_date)].copy()

upcoming_tba_df = netflix_df[(netflix_df['Premiere'].isna()) | (netflix_df['Premiere'] >= cutoff_date)].copy()

In [73]:
premiered_df.to_csv('../data/series/raw/netflix_titles.csv', index=False)
upcoming_tba_df.to_csv('../data/series/raw/upcoming_netflix.csv', index=False)

In [74]:
print(f"Premiered titles: {len(premiered_df)}")
print(f"Upcoming/TBA titles: {len(upcoming_tba_df)}")

Premiered titles: 300
Upcoming/TBA titles: 232


## Retrieving IMDb Metadata

In [75]:
import pandas as pd
import requests

API_KEY = "adebd275"

def get_omdb_data(title):
    try:
        url = f"https://www.omdbapi.com/?t={title}&apikey={API_KEY}"

        data = requests.get(url).json()

        return pd.Series({
            "Rated": data.get("Rated"),
            "imdb_rating": data.get("imdbRating"),
            "imdb_votes": data.get("imdbVotes"),
            "imdb_genre": data.get("Genre"),
            "imdb_year": data.get("Year"),
            "imdb_type": data.get("Type")
        })

    except:
        return pd.Series()

In [76]:
netflix_df = pd.read_csv('../data/series/raw/netflix_titles.csv')

In [77]:
omdb_data = netflix_df["Title"].apply(get_omdb_data)

netflix_df = pd.concat([netflix_df, omdb_data], axis=1)

netflix_df.head()

,Title,Premiere,Rated,imdb_rating,imdb_votes,imdb_genre,imdb_year,imdb_type
0,Virgin River,2019-12-06,TV-14,7.4,"57,217","Drama, Romance",2019–,series
1,The Witcher,2019-12-20,TV-MA,7.9,"618,310","Action, Adventure, Drama",2019–,series
2,Outer Banks,2020-04-15,TV-MA,7.5,"104,906","Action, Crime, Drama",2020–2026,series
3,Sweet Magnolias,2020-05-19,TV-14,7.3,"27,427","Drama, Romance",2020–,series
4,Bridgerton,2020-12-25,TV-MA,7.5,"219,043","Drama, Romance",2020–,series


In [78]:
netflix_df.shape

(300, 8)

In [79]:
netflix_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Title        300 non-null    str  
 1   Premiere     300 non-null    str  
 2   Rated        277 non-null    str  
 3   imdb_rating  277 non-null    str  
 4   imdb_votes   277 non-null    str  
 5   imdb_genre   277 non-null    str  
 6   imdb_year    277 non-null    str  
 7   imdb_type    277 non-null    str  
dtypes: str(8)
memory usage: 18.9 KB


In [80]:
netflix_df['imdb_votes'] = (netflix_df['imdb_votes'].astype(str).str.replace(',', '', regex=False))

In [81]:
netflix_df['imdb_rating'] = pd.to_numeric(netflix_df['imdb_rating'], errors='coerce')
netflix_df['imdb_votes'] = pd.to_numeric(netflix_df['imdb_votes'], errors='coerce')

In [82]:
netflix_df['imdb_year'] = (netflix_df['imdb_year'].astype(str).str.extract(r'(\d{4})')[0].astype('Int64'))

In [83]:
netflix_df['imdb_year'] = pd.to_numeric(netflix_df['imdb_year'], errors='coerce')

In [84]:
netflix_df = netflix_df.dropna(subset=['imdb_rating'])

In [85]:
netflix_df.info()

<class 'pandas.DataFrame'>
Index: 222 entries, 0 to 296
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Title        222 non-null    str    
 1   Premiere     222 non-null    str    
 2   Rated        222 non-null    str    
 3   imdb_rating  222 non-null    float64
 4   imdb_votes   222 non-null    float64
 5   imdb_genre   222 non-null    str    
 6   imdb_year    222 non-null    Int64  
 7   imdb_type    222 non-null    str    
dtypes: Int64(1), float64(2), str(5)
memory usage: 15.8 KB


In [ ]:
netflix_df.to_csv("../data/series/processed/netflix_imdb.csv",index=False)

## Adaptation Verification and Saving the Dataset

In [87]:
import pandas as pd

df = pd.read_csv("../data/series/processed/netflix_imdb_adaptations.csv")

df.head()

,Title,Premiere,Rated,imdb_rating,imdb_votes,imdb_genre,imdb_year,imdb_type,is_book_adaptation,book_title,book_author
0,Virgin River,06/12/2019,TV-14,7.4,57217,"Drama, Romance",2019,series,y,Virgin River series,Robyn Carr
1,The Witcher,20/12/2019,TV-MA,7.9,618310,"Action, Adventure, Drama",2019,series,y,The Witcher series,Andrzej Sapkowski
2,Outer Banks,15/04/2020,TV-MA,7.5,104906,"Action, Crime, Drama",2020,series,n,NaN,NaN
3,Sweet Magnolias,19/05/2020,TV-14,7.3,27427,"Drama, Romance",2020,series,y,Sweet Magnolias series,Sherryl Woods
4,Bridgerton,25/12/2020,TV-MA,7.5,219043,"Drama, Romance",2020,series,y,Bridgerton series,Julia Quinn


In [88]:
df["is_book_adaptation"].value_counts()

is_book_adaptation
n    184
y     38
Name: count, dtype: int64

In [89]:
adaptations_df = df[df["is_book_adaptation"] == "y"].copy()

In [90]:
adaptations_df.shape

(38, 11)

In [91]:
adaptations_df[["book_title", "book_author"]].isnull().sum()

book_title     0
book_author    0
dtype: int64

In [92]:
adaptations_df["book_title"] = adaptations_df["book_title"].str.strip().str.lower()
adaptations_df["book_author"] = adaptations_df["book_author"].str.strip().str.lower()

In [93]:
adaptations_df[["book_title", "book_author"]].head(10)

,book_title,book_author
0,virgin river series,robyn carr
1,the witcher series,andrzej sapkowski
3,sweet magnolias series,sherryl woods
4,bridgerton series,julia quinn
5,the brass verdict,michael connelly
6,the night agent,matthew quirk
9,my life with the walter boys,ali novak
10,the three-body problem (remembrance of earth's...,liu cixin
14,ransom canyon series,jodi thomas
15,forever…,judy blume


In [94]:
import re

def clean_title(text):
    text = text.lower()
    text = re.sub(r"\(.*?\)", "", text)   # remove brackets content
    text = text.replace(",", " ")
    text = re.sub(r"\s+", " ", text)     # remove extra spaces
    return text.strip()

adaptations_df["book_title"] = adaptations_df["book_title"].apply(clean_title)

In [95]:
def to_search_key(text):
    text = text.lower()
    text = re.sub(r"(series|universe|novels|books)", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

adaptations_df["search_key"] = adaptations_df["book_title"].apply(to_search_key)

In [96]:
adaptations_df.head(10)

,Title,Premiere,Rated,imdb_rating,imdb_votes,imdb_genre,imdb_year,imdb_type,is_book_adaptation,book_title,book_author,search_key
0,Virgin River,06/12/2019,TV-14,7.4,57217,"Drama, Romance",2019,series,y,virgin river series,robyn carr,virgin river
1,The Witcher,20/12/2019,TV-MA,7.9,618310,"Action, Adventure, Drama",2019,series,y,the witcher series,andrzej sapkowski,the witcher
3,Sweet Magnolias,19/05/2020,TV-14,7.3,27427,"Drama, Romance",2020,series,y,sweet magnolias series,sherryl woods,sweet magnolias
4,Bridgerton,25/12/2020,TV-MA,7.5,219043,"Drama, Romance",2020,series,y,bridgerton series,julia quinn,bridgerton
5,The Lincoln Lawyer,13/05/2022,R,7.3,266302,"Crime, Drama, Mystery",2011,movie,y,the brass verdict,michael connelly,the brass verdict
6,The Night Agent,23/03/2023,TV-MA,7.4,151290,"Action, Drama, Thriller",2023,series,y,the night agent,matthew quirk,the night agent
9,My Life with the Walter Boys,07/12/2023,TV-14,6.8,22962,"Drama, Romance",2023,series,y,my life with the walter boys,ali novak,my life with the walter boys
10,3 Body Problem,21/03/2024,TV-MA,7.5,177200,"Adventure, Drama, Fantasy",2024,series,y,the three-body problem,liu cixin,the three-body problem
14,Ransom Canyon,17/04/2025,TV-MA,6.8,9381,"Drama, Romance, Western",2025,series,y,ransom canyon series,jodi thomas,ransom canyon
15,Forever,08/05/2025,TV-PG,8.2,65147,"Crime, Drama, Fantasy",2014,series,y,forever…,judy blume,forever…


In [97]:
adaptations_df.duplicated().sum()

np.int64(0)

In [98]:
mismatch_df = adaptations_df[pd.to_datetime(adaptations_df['Premiere'], dayfirst=True).dt.year != adaptations_df['imdb_year']]

print(len(mismatch_df))

7


In [99]:
mismatch_df['year_diff'] = (pd.to_datetime(mismatch_df['Premiere'], dayfirst=True).dt.year - mismatch_df['imdb_year']).abs()

print(mismatch_df['year_diff'].value_counts().sort_index())

year_diff
2     1
7     1
11    3
22    1
24    1
Name: count, dtype: int64


In [100]:
mismatch_df = mismatch_df.sort_values('year_diff', ascending=False)

mismatch_df.to_csv('../data/series/processed/netflix_year_mismatches.csv', index=False)

print(f"Saved {len(mismatch_df)} mismatches")

Saved 7 mismatches


In [101]:
mismatch_df = pd.read_csv('../data/series/processed/corrected_netflix_mismatches.csv')

print("Remove:", (mismatch_df['keep'].str.lower() == 'n').sum())
print("Update:", (mismatch_df['keep'].str.lower() == 'y').sum())

Remove: 0
Update: 7


In [102]:
titles_to_remove = mismatch_df.loc[mismatch_df['keep'].str.lower() == 'n','Title']

adaptations_df = adaptations_df[~adaptations_df['Title'].isin(titles_to_remove)].copy()

In [103]:
updates = mismatch_df[mismatch_df['keep'].str.lower() == 'y'].set_index('Title')

adaptations_df = adaptations_df.set_index('Title')

cols_to_update = [
    'Rated',
    'imdb_rating',
    'imdb_votes',
    'imdb_genre',
    'imdb_year',
    'imdb_type'
]

for col in cols_to_update:
    adaptations_df.loc[updates.index, col] = updates[col]

adaptations_df = adaptations_df.reset_index()

In [104]:
print("Final rows:", len(adaptations_df))

Final rows: 38


In [105]:
adaptations_df.to_csv("../data/series/final/netflix_series.csv", index=False)